In [4]:
import os
from dotenv import load_dotenv
from unstructured.partition.pdf import partition_pdf

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGCHAIN_TRACING_V2 = "true"


output_path = "../output/"
file_path = "../content/OGUC.pdf"

In [5]:
from unstructured.partition.pdf import partition_pdf

# Reference: https://docs.unstructured.io/open-source/core-functionality/chunking
chunks = partition_pdf(
    filename=file_path,
    infer_table_structure=True,            # extract tables
    strategy="hi_res",                     # mandatory to infer tables

    extract_image_block_types=["Image"],   # Add 'Table' to list to extract image of tables
    # image_output_dir_path=output_path,   # if None, images and tables will saved in base64

    extract_image_block_to_payload=True,   # if true, will extract base64 for API usage

    chunking_strategy="by_title",          # or 'basic'
    max_characters=10000,                  # defaults to 500
    combine_text_under_n_chars=2000,       # defaults to 0
    new_after_n_chars=6000,

    # extract_images_in_pdf=True,          # deprecated
)

In [3]:
len(chunks)

35

In [6]:
# separate tables from texts
tables = []
texts = []

for chunk in chunks:
    texts.append(chunk)
    chunk_elements = chunk.metadata.orig_elements
    for el in chunk_elements:
        if "Table" in str(type(el)):
            
            tables.append(el)

print(len(texts))
print(len(tables))

356
79


In [7]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_text = """
Eres un asistente que resume tablas y texto.
Entrega un resumen conciso de la tabla o texto.

No comiences tu mensaje diciendo "Aqui hay un resumen" o algo similar.
Simplemente entrega el resumen como tal.

Tabla o texto: {element}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)

# Summary chain
model = ChatGroq(temperature=0.5, model="llama-3.1-8b-instant")
summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()

In [10]:
import asyncio
from datetime import datetime, timedelta
from collections import deque
import time
from typing import List, Any
import nest_asyncio

class RateLimiter:
    def __init__(self):
        self.rpm_limit = 1000  # Requests per minute
        self.tpm_limit = 250000  # Tokens per minute
        self.rpd_limit = 500000  # Requests per day
        
        # Ventanas deslizantes para tracking
        self.minute_requests = deque(maxlen=1000)
        self.minute_tokens = deque(maxlen=250000)
        self.day_requests = deque(maxlen=500000)
        
        # Tiempo mínimo entre requests (60 seg / 1000 rpm = 0.06 seg)
        self.min_interval = 60 / self.rpm_limit

    async def wait_if_needed(self, tokens: int = 1):
        current_time = datetime.now()
        self._clean_old_records(current_time)
        
        while (
            len(self.minute_requests) >= self.rpm_limit or
            len(self.minute_tokens) + tokens >= self.tpm_limit or
            len(self.day_requests) >= self.rpd_limit
        ):
            await asyncio.sleep(0.1)
            current_time = datetime.now()
            self._clean_old_records(current_time)
        
        self.minute_requests.append(current_time)
        self.day_requests.append(current_time)
        for _ in range(tokens):
            self.minute_tokens.append(current_time)

    def _clean_old_records(self, current_time: datetime):
        minute_ago = current_time - timedelta(minutes=1)
        day_ago = current_time - timedelta(days=1)
        
        while (self.minute_requests and 
               self.minute_requests[0] < minute_ago):
            self.minute_requests.popleft()
            
        while (self.minute_tokens and 
               self.minute_tokens[0] < minute_ago):
            self.minute_tokens.popleft()
            
        while (self.day_requests and 
               self.day_requests[0] < day_ago):
            self.day_requests.popleft()

async def process_batch_with_rate_limit(items: List[Any], 
                                      process_func, 
                                      batch_size: int = 3,
                                      tokens_per_request: int = 1000):
    """
    Procesa items en batches respetando rate limits
    """
    rate_limiter = RateLimiter()
    results = []
    
    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]
        batch_tasks = []
        
        for item in batch:
            await rate_limiter.wait_if_needed(tokens_per_request)
            task = asyncio.create_task(process_func(item))
            batch_tasks.append(task)
        
        batch_results = await asyncio.gather(*batch_tasks)
        results.extend(batch_results)
        
    return results

def prepare_content(texts, tables, limit=None):
    """
    Prepara el contenido de textos y tablas para procesamiento
    limit: int o None. Si es None, procesa todos los elementos
    """
    # Si limit es None, usar toda la lista
    text_contents = [text.text for text in (texts[:limit] if limit else texts)]
    
    tables_content = []
    for table in (tables[:limit] if limit else tables):
        table_html = table.metadata.text_as_html 
        table_text = table.text
        element = table_html + "\n\n" + table_text
        tables_content.append(element)
        
    return text_contents, tables_content

async def process_summaries(texts, tables, limit=None):
    """
    Procesa los resúmenes de textos y tablas
    Retorna: tuple (text_summaries: list, table_summaries: list)
    """
    text_contents, tables_content = prepare_content(texts, tables, limit)
    
    print(f"Procesando {len(text_contents)} textos y {len(tables_content)} tablas")
    
    # Procesar textos
    text_summaries = await process_batch_with_rate_limit(
        items=text_contents,
        process_func=lambda x: summarize_chain.ainvoke({"element": x}),
        batch_size=3,
        tokens_per_request=1000
    )

    # Procesar tablas
    table_summaries = await process_batch_with_rate_limit(
        items=tables_content,
        process_func=lambda x: summarize_chain.ainvoke({"element": x}),
        batch_size=3,
        tokens_per_request=1000
    )
    
    return text_summaries, table_summaries

# Aplicar nest_asyncio para Jupyter
nest_asyncio.apply()

# Ejecutar el procesamiento
async def main():
    text_summaries, table_summaries = await process_summaries(texts, tables, limit=None)
    
    # Imprimir resultados
    print("Resúmenes de Texto:")
    for i, summary in enumerate(text_summaries):
        print(f"\nTexto {i + 1}:")
        print(summary)

    print("\nResúmenes de Tablas:")
    for i, summary in enumerate(table_summaries):
        print(f"\nTabla {i + 1}:")
        print(summary)
    
    return text_summaries, table_summaries

# Ejecutar
text_summaries, table_summaries = await main()


Procesando 356 textos y 79 tablas
Resúmenes de Texto:

Texto 1:
La Ordenanza General de Urbanismo y Construcciones ha sido modificada y rectificada en varias ocasiones desde su publicación en 1992. Las modificaciones incluyen:

- Adición de artículos y capítulos sobre temas como la resistencia al fuego, terminales de locomoción colectiva, locales escolares y vivienda rural.
- Modificaciones a la aplicación de coeficientes constructibilidad y acondicionamiento térmico de techumbres.
- Rectificaciones y anulaciones de artículos y capítulos por errores formales o inconsistencias.
- Inclusión de normas de seguridad contra incendios y otras medidas para facilitar el desplazamiento de personas con discapacidad.
- Modificaciones a la calidad de construcción y responsabilidades de los propietarios.
- Sustitución y agregado de definiciones de vocablos y reemplazo de capítulos.

La Ordenanza General ha sido actualizada en varias ocasiones desde su publicación inicial en 1992.

Texto 2:
La Ordena

In [14]:
table_summaries

['La tabla muestra una lista de decretos supremos (D.S.) emitidos por la autoridad competente, relacionados con la normativa urbanística y de construcción en Chile. La información incluye el número de decreto, fecha de emisión, vigencia, materia y descripción de los cambios realizados en la normativa.\n\nLos D.S. abarcan temas como la publicación de ordenanzas generales, la rectificación de errores formales, la modificación de conjuntos armonicos, la aprobación de planes reguladores comunales, la seguridad contra incendios, la calidad de construcción y responsabilidades, entre otros.\n\nLa fecha de emisión de los D.S. varía desde mayo de 1992 hasta abril de 2001, con un total de 45 decretos emitidos. La vigencia de algunos de los D.S. es hasta 2000, mientras que otros tienen una vigencia indefinida.',
 'La tabla muestra una lista de disposiciones supremas (D.S.) con sus fechas de vigencia y materia. Los temas tratados incluyen:\n\n- Modificaciones a la Ordenanza General D.S. N°47/92\n-

In [13]:
len(table_summaries)

79

In [15]:
import uuid
from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain.retrievers.multi_vector import MultiVectorRetriever

vectorstore = Chroma(collection_name="multi_modal_rag", embedding_function=OpenAIEmbeddings())

store = InMemoryStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)


/tmp/ipykernel_118338/3275223840.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(collection_name="multi_modal_rag", embedding_function=OpenAIEmbeddings())


In [16]:
# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=summary, metadata={id_key: doc_ids[i]}) for i, summary in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=summary, metadata={id_key: table_ids[i]}) for i, summary in enumerate(table_summaries)
]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, tables)))

In [17]:
chunks = retriever.invoke(
    "que dice la ordenanza con respecto a los estacionamientos?"
)

print(type(chunks))

<class 'list'>


In [18]:
chunks

In [19]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from base64 import b64decode


def parse_docs(docs):
    """Split base64-encoded images and texts"""
    b64 = []
    text = []
    for doc in docs:
        try:
            b64decode(doc)
            b64.append(doc)
        except Exception as e:
            text.append(doc)
    return {"images": b64, "texts": text}


def build_prompt(kwargs):

    docs_by_type = kwargs["context"]
    user_question = kwargs["question"]

    context_text = ""
    if len(docs_by_type["texts"]) > 0:
        for text_element in docs_by_type["texts"]:
            context_text += text_element.text

    # construct prompt with context (including images)
    prompt_template = f"""
    Answer the question based only on the following context, which can include text, tables, and the below image.
    Context: {context_text}
    Question: {user_question}
    """

    prompt_content = [{"type": "text", "text": prompt_template}]

    if len(docs_by_type["images"]) > 0:
        for image in docs_by_type["images"]:
            prompt_content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image}"},
                }
            )

    return ChatPromptTemplate.from_messages(
        [
            HumanMessage(content=prompt_content),
        ]
    )


chain = (
    {
        "context": retriever | RunnableLambda(parse_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(build_prompt)
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

chain_with_sources = {
    "context": retriever | RunnableLambda(parse_docs),
    "question": RunnablePassthrough(),
} | RunnablePassthrough().assign(
    response=(
        RunnableLambda(build_prompt)
        | ChatOpenAI(model="gpt-4o-mini")
        | StrOutputParser()
    )
)

In [24]:
response = chain_with_sources.invoke(
    "Cual es el articulo de la oguc que habla sobre usos de suelo industriales?"
)

print("Response:", response['response'])

print("\n\nContext:")
for text in response['context']['texts']:
    print(text.text)
    print("Page number: ", text.metadata.page_number)
    print("\n" + "-"*50 + "\n")

Response: El artículo de la OGUC que habla sobre usos de suelo industriales es el **Artículo 2.1.28**, que se refiere al tipo de uso "Actividades Productivas".


Context:
ABRIL 2023

ANEXO 1-19

ARTICULOS TRANSITORIOS Y OTRAS MATERIAS RELACIONADAS CON LA ORDENANZA GENERAL DE URBANISMO Y CONSTRUCCIONES

Cursa con alcance el D.S. N°57, de 2018 del Ministerio de Vivienda y Urbanismo por N°E325313, de 23 de marzo de 2023, de la Contraloría General de la República.

La Contraloría General ha dado curso al instrumento de la suma, que modifica el decreto N°47, de 1992, del Ministerio de Vivienda y Urbanismo, Ordenanza General de Urbanismo y Construcciones (OGUC), con el objeto de adecuar sus normas a la ley N° 21.078, sobre Transparencia del Mercado del Suelo e Impuesto al Aumento de Valor por Ampliación del Límite Urbano y a la ley N° 21.074, sobre Fortalecimiento de la Regionalización del País.

No obstante, cumple con hacer presente que la referencia efectuada en la letra d) del numeral 1 